In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, Reshape
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
corpus = [
    "machine learning is fun",
    "deep learning is powerful",
    "artificial intelligence and machine learning",
    "neural networks are part of deep learning",
    "machine learning helps in data science"
]

In [ ]:
words = " ".join(corpus).lower().split()

vocab = list(set(words))
word_to_index = {word:i for i,word in enumerate(vocab)}
index_to_word = {i:word for word,i in word_to_index.items()}

vocab_size = len(vocab)
print("Vocabulary:", vocab)

Vocabulary: ['helps', 'fun', 'artificial', 'intelligence', 'machine', 'part', 'science', 'networks', 'learning', 'deep', 'data', 'are', 'powerful', 'neural', 'in', 'and', 'of', 'is']


In [ ]:
window_size = 2
training_data = []

for sentence in corpus:
    sentence_words = sentence.split()

    for i, target_word in enumerate(sentence_words):
        target_index = word_to_index[target_word]

        start = max(0, i-window_size)
        end = min(len(sentence_words), i+window_size+1)

        for j in range(start, end):
            if i != j:
                context_word = sentence_words[j]
                context_index = word_to_index[context_word]
                training_data.append([target_index, context_index])

training_data = np.array(training_data)

X = training_data[:,0]
y = training_data[:,1]

In [ ]:
embedding_dim = 10

input_layer = Input(shape=(1,))
embedding = Embedding(vocab_size, embedding_dim)(input_layer)
reshape = Reshape((embedding_dim,))(embedding)
output = Dense(vocab_size, activation='softmax')(reshape)

model = Model(input_layer, output)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 1, 10)          │           180 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 18)             │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 378 (1.48 KB)

 Trainable params: 378 (1.48 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 2.8881 
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8844
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8815
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8785
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8756
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 2.8730
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8701
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8674
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8645
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8617
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8590
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8561
Epoch 13/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2.8534
Epoch 14/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.8506
Epoch 15/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 2.8478 
Epoch 16/200
3/3 

In [ ]:
embeddings = model.layers[1].get_weights()[0]

print("Embedding vector for 'machine':")
print(embeddings[word_to_index['machine']])

Embedding vector for 'machine':
[-0.39098626  0.3422262  -0.39191175 -0.04773239 -0.4005816  -0.296118
  0.7257655  -0.33788493 -0.01054759 -0.08589803]


In [ ]:
def find_similar_words(word):

    word_vector = embeddings[word_to_index[word]].reshape(1,-1)

    similarities = cosine_similarity(word_vector, embeddings)[0]

    similar_words = sorted(
        [(index_to_word[i], similarities[i]) for i in range(len(similarities))],
        key=lambda x: x[1],
        reverse=True
    )

    return similar_words[1:6]

In [ ]:
print("Words similar to 'learning':")
print(find_similar_words("learning"))

print("Words similar to 'machine':")
print(find_similar_words("machine"))

Words similar to 'learning':
[('is', np.float32(0.544052)), ('helps', np.float32(0.25963628)), ('fun', np.float32(0.22981852)), ('powerful', np.float32(0.2242809)), ('part', np.float32(0.14251259))]
Words similar to 'machine':
[('deep', np.float32(0.7763191)), ('fun', np.float32(0.7537052)), ('powerful', np.float32(0.73991597)), ('and', np.float32(0.520832)), ('artificial', np.float32(0.42573518))]
